In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, dayofweek, hour, to_timestamp
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder
from pyspark.ml import Pipeline
# Initialisation (si pas déjà fait)
spark = SparkSession.builder.appName("LogisticsPrediction").getOrCreate()

path_data='././data/raw/DataCoSupplyChainDataset.csv'
df=spark.read.format("csv")\
    .option('header',"True")\
    .option("inferSchema","True")\
    .load(path_data)
print(f"Données chargées : {df.count()} lignes, {len(df.columns)} colonnes.")
# 1. Chargement des données (Simulation)
# df = spark.read.csv("votre_fichier.csv", header=True, inferSchema=True)

# 2. Sélection des colonnes (Top 10 + Target)
cols_to_keep = [
    "Late_delivery_risk",          # TARGET
    "Type",                        # Feature 1
    "Days for shipment (scheduled)", # Feature 2
    "Shipping Mode",               # Feature 3
    "Order Region",                # Feature 4
    "Order City",                  # Feature 5
    "Category Name",               # Feature 6
    "Customer Segment",            # Feature 7
    "Order Item Quantity",         # Feature 8
    "Order Item Total",            # Feature 9
    "Department Name"              # Feature 10
]

df_filtered = df.select(cols_to_keep)

# 3. Nettoyage (Suppression des NULLs éventuels)
df_cleaned = df_filtered.dropna()

# 4. Préparation du Pipeline de transformation (String -> Index -> Vector)
# Les colonnes catégoriques doivent être converties en nombres
categorical_cols = [
    "Type", "Shipping Mode", "Order Region", 
    "Order City", "Category Name", "Customer Segment", "Department Name"
]

# Feature Engineering Stages
stages = []

# A. Indexation des colonnes catégoriques (String -> Nombre)
for categorical_col in categorical_cols:
    string_indexer = StringIndexer(inputCol=categorical_col, outputCol=categorical_col + "_Index")
    # On ajoute l'encoder pour éviter que le modèle pense que Ville 2 > Ville 1 (Optionnel mais recommandé)
    encoder = OneHotEncoder(inputCols=[string_indexer.getOutputCol()], outputCols=[categorical_col + "_Vec"])
    stages += [string_indexer, encoder]

# B. Assemblage final des features en un seul vecteur
numeric_cols = ["Days for shipment (scheduled)", "Order Item Quantity", "Order Item Total"]
assembler_inputs = [c + "_Vec" for c in categorical_cols] + numeric_cols

assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages += [assembler]

# 5. Création et exécution du Pipeline
pipeline = Pipeline(stages=stages)
pipeline_model = pipeline.fit(df_cleaned)
df_transformed = pipeline_model.transform(df_cleaned)

# 6. Résultat final prêt pour l'entraînement
final_data = df_transformed.select("features", col("Late_delivery_risk").alias("label"))

print("Aperçu des données prêtes pour le modèle :")
final_data.show(5)

/usr/local/lib/python3.10/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/15 22:49:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/15 22:49:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Données chargées : 180519 lignes, 53 colonnes.


Aperçu des données prêtes pour le modèle :
+--------------------+-----+
|            features|label|
+--------------------+-----+
|(3688,[0,3,11,195...|    0|
|(3688,[1,3,15,211...|    1|
|(3688,[3,15,2115,...|    0|
|(3688,[0,3,9,324,...|    0|
|(3688,[2,3,9,324,...|    0|
+--------------------+-----+
only showing top 5 rows



In [2]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.sql.types import FloatType
import pyspark.sql.functions as F

# 1. Séparation des données (Train/Test Split)
# 80% pour l'entraînement, 20% pour tester la performance
train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)

print(f"Données d'entraînement : {train_data.count()} lignes")
print(f"Données de test : {test_data.count()} lignes")

# 2. Définition des 3 Modèles
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=20, maxDepth=5, seed=42)
gbt = GBTClassifier(featuresCol="features", labelCol="label", maxIter=20, seed=42) 
# Note: GBT est souvent le plus performant mais le plus long à entraîner

models = {
    "Logistic Regression": lr,
    "Random Forest": rf,
    "Gradient Boosted Trees": gbt
}

# 3. Boucle d'entraînement et d'évaluation
results = {}

print("\n--- Démarrage de l'entraînement comparatif ---\n")

for model_name, model in models.items():
    print(f"Entraînement de : {model_name}...")
    
    # A. Entraînement (Fit)
    trained_model = model.fit(train_data)
    
    # B. Prédiction sur les données de test
    predictions = trained_model.transform(test_data)
    
    # C. Calcul des Métriques
    # Evaluator pour l'AUC (Area Under ROC) - mesure la capacité à distinguer 0 et 1
    binary_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
    auc = binary_evaluator.evaluate(predictions)
    
    # Evaluator pour Accuracy, F1, Weighted Precision/Recall
    # F1-Score est crucial si vos classes sont déséquilibrées (ex: peu de retards)
    multi_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
    
    accuracy = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "accuracy"})
    f1_score = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "f1"})
    weighted_precision = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "weightedPrecision"})
    weighted_recall = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "weightedRecall"})
    
    # Stockage des résultats
    results[model_name] = {
        "AUC": auc,
        "Accuracy": accuracy,
        "F1 Score": f1_score,
        "Precision": weighted_precision,
        "Recall": weighted_recall
    }
    
    print(f"  -> Terminé. Accuracy: {accuracy:.2%}, AUC: {auc:.3f}")

# 4. Affichage du Tableau Comparatif Final
print("\n" + "="*65)
print(f"{'Modèle':<25} | {'Accuracy':<10} | {'AUC':<8} | {'F1 Score':<10}")
print("="*65)

best_model_name = ""
best_metric = 0

for name, metrics in results.items():
    print(f"{name:<25} | {metrics['Accuracy']:.2%}    | {metrics['AUC']:.3f}    | {metrics['F1 Score']:.3f}")
    
    # Logique simple pour trouver le 'meilleur' basé sur l'AUC
    if metrics['AUC'] > best_metric:
        best_metric = metrics['AUC']
        best_model_name = name

print("="*65)
print(f"\n🏆 Le modèle recommandé est : {best_model_name} avec une AUC de {best_metric:.3f}")

Données d'entraînement : 144625 lignes


Données de test : 35894 lignes

--- Démarrage de l'entraînement comparatif ---

Entraînement de : Logistic Regression...


25/11/15 22:49:52 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


  -> Terminé. Accuracy: 71.25%, AUC: 0.780
Entraînement de : Random Forest...


25/11/15 22:50:07 WARN MemoryStore: Not enough space to cache rdd_257_12 in memory! (computed 16.1 MiB so far)
25/11/15 22:50:07 WARN MemoryStore: Not enough space to cache rdd_257_11 in memory! (computed 16.1 MiB so far)
25/11/15 22:50:07 WARN MemoryStore: Not enough space to cache rdd_257_7 in memory! (computed 16.1 MiB so far)
25/11/15 22:50:07 WARN MemoryStore: Not enough space to cache rdd_257_9 in memory! (computed 24.6 MiB so far)
25/11/15 22:50:07 WARN MemoryStore: Not enough space to cache rdd_257_20 in memory! (computed 16.1 MiB so far)
25/11/15 22:50:07 WARN MemoryStore: Not enough space to cache rdd_257_5 in memory! (computed 16.1 MiB so far)
25/11/15 22:50:07 WARN MemoryStore: Not enough space to cache rdd_257_2 in memory! (computed 16.1 MiB so far)
25/11/15 22:50:07 WARN MemoryStore: Not enough space to cache rdd_257_0 in memory! (computed 16.1 MiB so far)
25/11/15 22:50:07 WARN MemoryStore: Not enough space to cache rdd_257_15 in memory! (computed 16.1 MiB so far)
25/11/

  -> Terminé. Accuracy: 54.56%, AUC: 0.727
Entraînement de : Gradient Boosted Trees...


25/11/15 22:50:24 WARN MemoryStore: Not enough space to cache rdd_363_12 in memory! (computed 24.3 MiB so far)
25/11/15 22:50:24 WARN MemoryStore: Not enough space to cache rdd_363_9 in memory! (computed 24.3 MiB so far)
25/11/15 22:50:24 WARN MemoryStore: Not enough space to cache rdd_363_0 in memory! (computed 15.9 MiB so far)
25/11/15 22:50:24 WARN MemoryStore: Not enough space to cache rdd_363_1 in memory! (computed 24.3 MiB so far)
25/11/15 22:50:24 WARN MemoryStore: Not enough space to cache rdd_363_10 in memory! (computed 24.3 MiB so far)
25/11/15 22:50:24 WARN MemoryStore: Not enough space to cache rdd_363_15 in memory! (computed 24.3 MiB so far)
25/11/15 22:50:24 WARN MemoryStore: Not enough space to cache rdd_363_17 in memory! (computed 15.9 MiB so far)
25/11/15 22:50:24 WARN BlockManager: Persisting block rdd_363_15 to disk instead.
25/11/15 22:50:24 WARN MemoryStore: Not enough space to cache rdd_363_16 in memory! (computed 24.3 MiB so far)
25/11/15 22:50:24 WARN MemoryStor

  -> Terminé. Accuracy: 70.18%, AUC: 0.747

Modèle                    | Accuracy   | AUC      | F1 Score  
Logistic Regression       | 71.25%    | 0.780    | 0.712
Random Forest             | 54.56%    | 0.727    | 0.385
Gradient Boosted Trees    | 70.18%    | 0.747    | 0.696

🏆 Le modèle recommandé est : Logistic Regression avec une AUC de 0.780
